# Lab-2b: Fine-Tuning Data Preparation (Text Summarization)

**Persona:** Data Engineer &nbsp;|&nbsp; **Lab:** 2 - Data Prep

## Overview

This notebook prepares the **instruction-tuning dataset** consumed by the fine-tuning lab (`lab3-model-build/lab-3b-fine-tuning.ipynb`). Rather than preparing data inline in the training notebook, data prep is a standalone, reusable step that publishes a ready-to-train dataset to Amazon S3.

### What this notebook does

1. Loads the **Databricks Dolly 15k** dataset
2. Filters for **summarization** examples to build a domain-specific model
3. Splits into train (70%) / test (30%)
4. Writes `train.jsonl` and `test.jsonl`
5. Creates the instruction **prompt template** (`template.json`) required by SageMaker JumpStart instruction fine-tuning
6. Uploads all three files to S3 as a complete fine-tuning package

### The Dolly Dataset

The Databricks Dolly dataset contains ~15,000 instruction-following examples across categories such as summarization, question answering, information extraction, creative writing, and classification. We keep only the **summarization** category.

### Data Format for Instruction Tuning

```json
{
  "instruction": "Summarize the following text",
  "context": "[Long text to summarize]",
  "response": "[Expected summary]"
}
```

### Output contract (consumed by Lab 3b)

All files are uploaded to: `s3://<default-bucket>/<ProfileName>/dolly_dataset/`
- `train.jsonl` - training examples
- `test.jsonl` - held-out evaluation examples
- `template.json` - prompt template for instruction tuning

## Step 1: Setup and Install Dependencies

<div style="padding: 15px; background-color: #fff3cd; border-left: 5px solid #ffc107; color: #856404;">
<strong>⚠️ Important:</strong> The cell below installs libraries and restarts the kernel. After the restart, continue with the next cell.
</div>

In [ ]:
!pip install -U "sagemaker>=3.17,<4" "datasets>=4.4.1" --quiet
# restart kernel
import IPython
IPython.Application.instance().kernel.do_shutdown(True) #automatically restarts kernel

In [ ]:
import datasets
import sagemaker
from packaging import version
from importlib.metadata import version as pkg_version

datasets_version = datasets.__version__
sagemaker_version = pkg_version("sagemaker")
print(f"datasets version: {datasets_version}")
print(f"sagemaker version: {sagemaker_version}")

if version.parse(datasets_version) < version.parse("4.4.1"):
    print("⚠️ Warning: datasets version is below 4.4.1. Please run the previous cell again")
elif version.parse(sagemaker_version) < version.parse("3.17.0"):
    print("⚠️ Warning: sagemaker version is below 3.17. Please run the previous cell again")
else:
    print("✓ Versions OK")

In [ ]:
import boto3
import sagemaker
import json
from sagemaker.core.helper.session_helper import Session, get_execution_role

# Initialize SageMaker session
sess = Session()
role = get_execution_role()
region = sess.boto_region_name

# Use the account/region default SageMaker bucket: sagemaker-<region>-<account-id>.
# Lab 2b writes to this same default bucket, but under a different top-level prefix
# (bank-marketing-lab/...), so the two labs never collide.
# NOTE: Lab 3b must resolve to this SAME bucket so the fine-tuning job finds this data.

bucket = sess.default_bucket()

print(f"Amazon SageMaker role: {role}")
print(f"Amazon S3 bucket: {bucket}")
print(f"AWS Region: {region}")

In [ ]:
# Derive the Studio user profile name; used to namespace the dataset in S3.
# This must match the value computed in lab-3b so both notebooks agree on the S3 path.
with open('/opt/ml/metadata/resource-metadata.json', 'r') as f:
    profile_name = json.load(f)['UserProfileName']

profile_name = profile_name[0].upper() + profile_name[1:]
print(f"Profile name: {profile_name}")

## Step 2: Load and Prepare the Dataset

Load Dolly, keep only summarization examples, and create a 70/30 train/test split. The test split is held out to evaluate performance on unseen data in Lab 3b.

In [ ]:
from datasets import load_dataset

dolly_dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

summarization_dataset = dolly_dataset.filter(lambda example: example["category"] == "summarization")
summarization_dataset = summarization_dataset.remove_columns("category")

# Split dataset: 70% for training the model, 30% held out to evaluate performance on unseen data
train_and_test_dataset = summarization_dataset.train_test_split(test_size=0.3)

train_and_test_dataset["train"].to_json("train.jsonl")
train_and_test_dataset["test"].to_json("test.jsonl")

print(f"Train examples: {train_and_test_dataset['train'].num_rows}")
print(f"Test examples:  {train_and_test_dataset['test'].num_rows}")

In [ ]:
print("Sample training example:")
train_and_test_dataset["train"][0]

## Step 3: Create Prompt Template

### Why Prompt Templates Matter

A **prompt template** defines how inputs are structured for the model. This is critical because:
- The model was pre-trained with specific formatting conventions
- Consistent formatting improves model performance
- The same template must be used for training AND inference

SageMaker JumpStart instruction fine-tuning expects a `template.json` in the training data channel alongside `train.jsonl`. The template follows the **instruction-input-response** pattern.

In [ ]:
template = {
    "prompt": "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Input:\n{context}\n\n",
    "completion": " {response}",
}

with open("template.json", "w") as f:
    json.dump(template, f)

print("✓ Prompt template created (template.json)")

## Step 4: Upload the Fine-Tuning Dataset to S3

Upload `train.jsonl`, `test.jsonl`, and `template.json` to S3. These are the exact locations Lab 3b reads for fine-tuning and evaluation.

In [ ]:
from sagemaker.core.s3 import S3Uploader

# Complete fine-tuning package location (consumed by Lab 3b)
data_location = f"s3://{bucket}/{profile_name}/dolly_dataset"

train_path = "train.jsonl"
template_path = "template.json"
evaluation_path = "test.jsonl"

training_input_path = f"{data_location}/{train_path}"
eval_input_path = f"{data_location}/{evaluation_path}"

S3Uploader.upload(train_path, data_location)
S3Uploader.upload(template_path, data_location)
S3Uploader.upload(evaluation_path, data_location)

print("✓ Fine-tuning dataset uploaded")
print(f"  Data location:    {data_location}")
print(f"  Training input:   {training_input_path}")
print(f"  Evaluation input: {eval_input_path}")

In [ ]:
# Verify the uploaded objects
from sagemaker.core.s3 import S3Downloader

print("Objects in the fine-tuning dataset location:")
for obj in S3Downloader.list(data_location):
    print(f"  {obj}")

## Summary

You prepared the summarization fine-tuning dataset and published a complete, ready-to-train package to S3:

- `s3://<default-bucket>/<ProfileName>/dolly_dataset/train.jsonl`
- `s3://<default-bucket>/<ProfileName>/dolly_dataset/test.jsonl`
- `s3://<default-bucket>/<ProfileName>/dolly_dataset/template.json`

The path derives from the account/region **default SageMaker bucket** (`sagemaker-<region>-<account-id>`) and your Studio profile name. **Lab 3b** reads from this same location, so make sure Lab 3b resolves its `bucket` to the same default bucket (set `bucket = sess.default_bucket()` there as well).

Continue to `lab3-model-build/lab-3b-fine-tuning.ipynb` to fine-tune Llama 3.2 3B on this prepared data.